# GraphSAGE — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def sigmoid(x): return 1/(1+np.exp(-x))
def softmax(x):
    x=np.asarray(x,dtype=float); e=np.exp(x-x.max()); return e/e.sum()
def draw_graph(A, pos=None, values=None, title=''):
    A=np.asarray(A); n=A.shape[0]
    if pos is None:
        ang=np.linspace(0,2*np.pi,n,endpoint=False); pos=np.c_[np.cos(ang),np.sin(ang)]
    plt.figure(figsize=(4,4))
    for i in range(n):
        for j in range(i+1,n):
            if A[i,j]!=0: plt.plot([pos[i,0],pos[j,0]],[pos[i,1],pos[j,1]],c='0.75',lw=1+2*abs(A[i,j]))
    c=values if values is not None else np.arange(n)
    plt.scatter(pos[:,0],pos[:,1],c=c,s=320,cmap='viridis',edgecolor='k',zorder=3)
    for i,(x,y) in enumerate(pos): plt.text(x,y,str(i),ha='center',va='center',color='white',weight='bold')
    plt.axis('off'); plt.title(title)
print('setup ok')

## GraphSAGE learns how to aggregate sampled neighborhoods so it can handle new nodes
GraphSAGE is message passing with neighbor sampling and an explicit combine step. The examples compute full and sampled means, concatenate self and neighbor features, embed an unseen node, and show sampling variance.

In [ ]:
# 1) full mean aggregator for one node
X=np.array([[1.,0.],[0.,1.],[1.,1.],[2.,0.]]); neigh=[1,2,3]; mean=X[neigh].mean(0)
plt.figure(figsize=(5,3)); plt.bar(['feat0','feat1'],mean); plt.title('full neighbor mean for node 0')
assert np.allclose(mean,[1,2/3])

In [ ]:
# 2) sampled aggregation approximates the full mean
X=np.array([[1.,0.],[0.,1.],[1.,1.],[2.,0.]]); sample=[1,3]; smean=X[sample].mean(0)
plt.figure(figsize=(5,3)); plt.bar(['feat0','feat1'],smean); plt.title('sampled mean from two neighbors')
assert np.allclose(smean,[1,0.5])

In [ ]:
# 3) concatenate self feature with neighborhood feature
x_self=np.array([1.,0.]); neigh_mean=np.array([1.,2/3]); h=np.r_[x_self,neigh_mean]
plt.figure(figsize=(5,3)); plt.bar(range(4),h); plt.title('GraphSAGE concat vector')
assert np.allclose(h,[1,0,1,2/3])

In [ ]:
# 4) inductive new node: same aggregator, no retraining needed
X=np.array([[1.,0.],[0.,1.],[1.,1.],[2.,0.],[0.5,0.5]]); new_mean=X[[0,2]].mean(0); h=np.r_[X[4],new_mean]
plt.figure(figsize=(5,3)); plt.bar(range(4),h); plt.title('embedding ingredients for new node 4')
assert np.allclose(h,[0.5,0.5,1,0.5])

In [ ]:
# 5) sampling variance falls as sample size grows
vals=np.array([[0.,1.],[1.,1.],[2.,0.],[3.,0.]]); rng=np.random.default_rng(1)
stds=[]
for k in [1,2,3]: stds.append(np.std([vals[rng.choice(4,k,replace=False)].mean(0)[0] for _ in range(200)]))
plt.figure(figsize=(5,3)); plt.plot([1,2,3],stds,'-o'); plt.xlabel('sample size'); plt.ylabel('std of mean feat0'); plt.title('larger samples are steadier')
assert stds[2]<stds[0]